# HS-Code Multimodal PDF RAG Pipeline
### Tables · Text → ChromaDB → Open-Source LLM (T4-optimised)

**Stack:**
| Component | Library / Model |
|---|---|
| PDF parsing | `pdfplumber` |
| Embeddings | `BAAI/bge-small-en-v1.5` (CPU-friendly, fast) |
| Vector store | `ChromaDB` (persistent) |
| Answer generation | Gemini AI |

> **GPU requirement:** T4 (15 GB). All models selected to fit comfortably within this budget.


## 0 — Install dependencies

In [12]:
%%capture
!pip install -q \
    pdfplumber \
    chromadb \
    sentence-transformers \
    torch \
    pandas

print('✅ All packages installed')

### For runnning the notebook of Mac

In [2]:
# Install TensorFlow Metal plugin for macOS GPU acceleration
!pip install tensorflow-metal

## 1 — Imports & global config

In [13]:
import os, io, re, uuid, textwrap, warnings
from pathlib import Path
from typing import List, Dict, Any, Tuple

import torch
import pdfplumber
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

DEVICE = torch.device(
    'mps'
    if torch.backends.mps.is_available() 
    else 'cuda' if torch.cuda.is_available() 
    else 'cpu'
)

print(f'Device: {DEVICE}')

if DEVICE.type == 'mps':
    print("Using MPS device")
    print(f"{os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024.**3):.2f} GB")
elif DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print("Using CPU")

# ── configurable knobs ───────────────────────────────────────────────────────
CHUNK_SIZE   = 512     # characters per text chunk
CHUNK_OVERLAP = 64     # character overlap between chunks
TOP_K        = 5       # retrieved chunks per query
COLLECTION_NAME = 'hs_code_rag'
PERSIST_DIR  = '../content/db'
# ─────────────────────────────────────────────────────────────────────────────

Device: mps
Using MPS device
16.00 GB


## 2 — Load PDF

In [4]:
PDF_PATH = "../content/docs/itc_hs_2022.pdf"
print(f'\nLoaded: {PDF_PATH}  ({os.path.getsize(PDF_PATH)/1e3:.1f} KB)')


Loaded: ../content/docs/itc_hs_2022.pdf  (7892.2 KB)


## 3 — PDF Extraction: text, tables, images

We use two complementary libraries:
* **`pdfplumber`** — precise table detection and text extraction with bounding boxes.
* No Images in the PDF

In [5]:
import re
import pandas as pd
import pdfplumber


def get_static_col_boundaries(page):
    """
    Static x-boundaries for the HS tariff tables in this PDF.
    The layout is stable across the first 10 pages.
    """
    # These values are tuned to the actual PDF layout:
    # tariff | separator | description | unit | standard rate | pref
    return [0, 120, 165, 400, 436, 490, page.width]


def get_row_key(row_keys, y):
    # map y to an existing row if close enough
    for k in row_keys:
        if abs(y - k) <= 3:
            return k
    row_keys.append(y)
    return y


def clean_cell(value):
    return re.sub(r"\s+", " ", value or "").strip()

def is_line_table_content(tariff, heading, description, unit, std_rate, pref):
    
    # if tariff contain any other character than digits or whitespace or '*', mark pure_tariff as None
    if tariff and not re.match(r"^[\d\s\*]+$", tariff):
        pure_tariff = None
    else:
        pure_tariff = tariff
    
    # if heading contains any other character than "-" or whitespace, mark pure_heading as None
    if heading and not re.match(r"^[\-\s]+$", heading):
        pure_heading = None
    else:
        pure_heading = heading
    
    # if there is no tarrif and no heading, it is not table content
    if not pure_tariff and not pure_heading:
        return False

    # else it is a table content
    return True
    
def enhance_chapter_data(chapter_data):
    for section, schapter, details, rows in chapter_data:
        # remove page numbers from description
        for row in rows:
            if row[2]:
                row[2] = re.sub(r"[\d\s]+$", "", row[2]).strip()

def extract_chapters_data_from_pdf(pdf_path):
    extracted_data = []
    current_section = None
    current_chapter = None
    current_text = ""
    current_rows = []
    page_start_y = 60

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            
            page_text = page.extract_text() or ""

            # section match
            section_match = re.search(r"SECTION\s*-\s*([A-Z0-9IVXLCDM]+)", page_text, flags=re.I)
            if section_match:
                section_group = section_match.group(1)
                if section_group:
                    current_section = f"Section {section_group}"
                    
            # chapter detection
            chapter_match = re.search(r"CHAPTER-(\d+)", page_text, flags=re.I)
            if chapter_match:
                chapter_group = chapter_match.group(1)
                if chapter_group:
                    new_chapter = f"Chapter {chapter_group}"
                    if new_chapter != current_chapter:
                        extracted_data.append((current_section, current_chapter, current_text, current_rows))
                        current_chapter = new_chapter
                        current_text = ""
                        current_rows = []
                
            # x boundaries
            x_breaks = get_static_col_boundaries(page)

            # crop below the header row
            words = page.extract_words()
            marker_words = [w for w in words if w["text"].strip() in {"(1)", "(2)", "(3)", "(4)", "(5)"}]
            
            header_bottom_y = page_start_y
            if len(marker_words) == 5:
                header_bottom_y = max(w["bottom"] for w in marker_words)
                
            if header_bottom_y > page_start_y:
                extracted_page_text = page.crop((0, 0, page.width, header_bottom_y)).extract_text() or ""
                extracted_page_text = re.sub(r"\(1\) \(2\) \(3\) \(4\) \(5\)", "", extracted_page_text)
                extracted_page_text = re.sub(r"Tariff Item Description of goods Unit Rate of duty\nStandard Prefer-\nential\nAreas", " ", extracted_page_text)
                if not re.fullmatch(r"\s*SECTION\s*-\s*[A-Z0-9IVXLCDM]+\s+CHAPTER\s*-\s*\d+\s*", extracted_page_text, flags=re.I,):
                    current_text += extracted_page_text + "\n"
                
            
            crop = page.crop((0, header_bottom_y + 5, page.width, page.height - 15))

            words = crop.extract_words()
            if not words:
                continue

            row_map = {}
            row_keys = []

            for w in words:
                txt = w["text"].strip()
                if not txt or txt in {"(1)", "(2)", "(3)", "(4)", "(5)"}:
                    continue
                
                y = (w["top"] + w["bottom"]) / 2
                line_key = get_row_key(row_keys, y)
                    
                x0 = w["x0"]

                # fixed column assignment for this PDF
                if x0 < x_breaks[1]:
                    col = "tariff"
                elif x0 < x_breaks[2]:
                    col = "separator"
                elif x0 < x_breaks[3]:
                    col = "description"
                elif x0 < x_breaks[4]:
                    col = "unit"
                elif x0 < x_breaks[5]:
                    col = "std_rate"
                else:
                    col = "pref"

                row_map.setdefault(line_key, {
                    "tariff": [],
                    "separator": [],
                    "description": [],
                    "unit": [],
                    "std_rate": [],
                    "pref": []
                })
                row_map[line_key][col].append(txt)

            for line_key in sorted(row_map):
                cells = row_map[line_key]

                tariff = clean_cell(" ".join(cells["tariff"]))
                separator = clean_cell(" ".join(cells["separator"]))
                desc = clean_cell(" ".join(cells["description"]))
                unit = clean_cell(" ".join(cells["unit"]))
                std_rate = clean_cell(" ".join(cells["std_rate"]))
                pref = clean_cell(" ".join(cells["pref"]))
                    
                # if separator contains white space before, after or in-between "-"s, remove i
                separator = re.sub(r"\s*-\s*", "-", separator)

                # if no content
                if not any([tariff, separator, desc, unit, std_rate, pref]):
                    continue

                # skip chapter heading rows
                if re.match(r"^(SECTION|CHAPTER|Page)\b", " ".join([tariff, desc, unit, std_rate, pref]), flags=re.I):
                    continue
                
                # skip page number at the bottom of the page
                if re.match(r"^Page\b", " ".join([tariff, desc, unit, std_rate, pref]), flags=re.I):
                    continue

                # continuation description rows
                if (
                    not tariff
                    and not separator
                    and not unit
                    and not std_rate
                    and not pref
                    and desc
                    and current_rows
                ):
                    current_rows[-1][2] = clean_cell(f"{current_rows[-1][2]} {desc}")
                    continue

                # skip rows that are not table content
                if not is_line_table_content(tariff, separator, desc, unit, std_rate, pref):
                    line_text = " ".join([tariff, separator, desc, unit, std_rate, pref])
                    current_text = current_text + "\n" + line_text
                    continue

                current_rows.append([tariff, separator, desc, unit, std_rate, pref])

        if current_chapter and current_text and current_rows:
            extracted_data.append((current_section, current_chapter, current_text, current_rows))
    
    enhance_chapter_data(extracted_data)           
    return extracted_data
        

chapters_data = extract_chapters_data_from_pdf(PDF_PATH)
headers = ["Tariff Item", "Headings", "Description of goods", "Unit", "Standard Rate", "Preferential Areas"]
     
for section_name, chapter_name, chapter_details, rows in chapters_data:
    df = pd.DataFrame(rows, columns=headers)
    print(f"{section_name} - {chapter_name} with {len(df)} rows")
         
    # Save dataframe to CSV
    df.to_csv(f"../content/chapters/{chapter_name}.csv", index=False)
    

Section I - None with 0 rows
Section I - Chapter 1 with 68 rows
Section I - Chapter 2 with 91 rows
Section I - Chapter 3 with 380 rows
Section I - Chapter 4 with 75 rows
Section II - Chapter 5 with 111 rows
Section II - Chapter 6 with 33 rows
Section II - Chapter 7 with 136 rows
Section II - Chapter 8 with 159 rows
Section II - Chapter 9 with 205 rows
Section II - Chapter 10 with 62 rows
Section II - Chapter 11 with 57 rows
Section II - Chapter 12 with 157 rows
Section II - Chapter 13 with 63 rows
Section III - Chapter 14 with 21 rows
Section IV - Chapter 15 with 197 rows
Section IV - Chapter 16 with 62 rows
Section IV - Chapter 17 with 62 rows
Section IV - Chapter 18 with 19 rows
Section IV - Chapter 19 with 47 rows
Section IV - Chapter 20 with 100 rows
Section IV - Chapter 21 with 57 rows
Section IV - Chapter 22 with 108 rows
Section IV - Chapter 23 with 89 rows
Section V - Chapter 24 with 68 rows
Section V - Chapter 25 with 249 rows
Section V - Chapter 26 with 110 rows
Section VI - 

In [6]:
def create_table_chunks(chapter_data):
    chunks = []
    for section, chapter, details, rows in chapter_data:
        if not rows:
            continue
        chunk_rows = []
        chunk_tariff = None
        chunk_description = None
        for row in rows:
            tariff = row[0]
            heading = row[1]
            description = row[2]
            header_tarrif = None
            if tariff and len(tariff) >= 4:
                header_tarrif = tariff[:4]
            if header_tarrif and header_tarrif != chunk_tariff:
                if chunk_tariff and chunk_description and len(chunk_rows) > 0:
                    chunk = {
                        "section": section,
                        "chapter": chapter,
                        "heading": chunk_tariff,
                        "description": chunk_description,
                        "rows": chunk_rows
                    }
                    chunks.append(chunk)
                chunk_tariff = header_tarrif
                chunk_description = description
                chunk_rows = []
            chunk_rows.append(row)
    return chunks

def create_text_chunks(chapters_data):
    chunks = []
    for section, chapter, details, rows in chapters_data:
        if not details:
            continue
        # chunk details into {CHUNK_SIZE} words chunks with section and chapter information
        # overlap chunk boundaries by {CHUNK_OVERLAP} words
        words = details.split()
        for i in range(0, len(words), CHUNK_SIZE-CHUNK_OVERLAP):
            chunk = {
                "section": section,
                "chapter": chapter,
                "details": " ".join(words[i:i+CHUNK_SIZE])
            }
            chunks.append(chunk)
    return chunks
        

table_chunks = create_table_chunks(chapters_data)
text_chunks = create_text_chunks(chapters_data)

print(f"Table chunks: {len(table_chunks)}")
print(f"Text chunks: {len(text_chunks)}")

Table chunks: 1184
Text chunks: 162


## 4 — Table verbalization

In [7]:
def table_to_markdown(rows: List[List]) -> str:
    """Convert extracted table rows to a Markdown string."""
    clean = []
    for row in rows:
        clean.append([str(cell).strip() if cell is not None else '' for cell in row])
    if not clean:
        return ''
    num_columns = len(clean[0])
    header = '| HS Code | Heading Indicator | Description | Unit | Standard Rate | Preference |'
    lines = [header]
    # lines  = ['| ' + ' | '.join(header) + ' |']
    lines += ['| ' + ' | '.join(['---'] * num_columns) + ' |']
    for row in clean:
        # pad row if shorter than header
        row += [''] * (num_columns - len(row))
        lines.append('| ' + ' | '.join(row[:num_columns]) + ' |')
    return '\n'.join(lines)


@torch.inference_mode()
def verbalize_table(table_chunk) -> str:
    """Verbalize a table chunk into a descriptive string with Markdown."""
    section = table_chunk['section']
    chapter = table_chunk['chapter']
    heading = table_chunk['heading']
    description = table_chunk['description']
    rows = table_chunk['rows']
    
    markdown = table_to_markdown(rows)
    if not markdown:
        return ''

    # store both the prose and the markdown for retrieval richness
    return f'{section} - {chapter} - {heading}\n\n{description}\n\n[Table Markdown]\n{markdown}\n'

for table_chunk in table_chunks:
    verbalized = verbalize_table(table_chunk)
    table_chunk['content'] = verbalized
    
print(f"Verbalized table chunks: {len(table_chunks)}")
    

Verbalized table chunks: 1184


In [8]:
print(text_chunks[4])

{'section': 'Section I', 'chapter': 'Chapter 4', 'details': 'CHAPTER 4 Dairy produce; birds’ eggs; natural honey; edible products of animal origin, not elsewhere specified or included NOTES : 1. The expression ―milk‖ means full cream milk or partially or completely skimmed milk. 2. For the purposes of heading 0403, yogurt may be concentrated or flavoured and may contain added sugar or other sweetening matter, fruit, nuts, cocoa, chocolate, spices, coffee or coffee extracts, plants, parts of plants, cereals or bakers‘ wares, provided that any added substance is not used for the purpose of replacing, in whole or in part, any milk constituent, and the product retains the essential character of yogurt. 3. For the purposes of heading 0405 : (a) the term ―butter‖ means natural butter, whey butter or recombined butter (fresh, salted or rancid, including canned butter) derived exclusively from milk, with a milkfat content of 80% or more but not more than 95% by weight, a maximum milk solids-no

In [9]:
print(table_chunks[0]['content'])

Section I - Chapter 1 - 0101

LIVE HORSES, ASSES, MULES AND HINNIES

[Table Markdown]
| HS Code | Heading Indicator | Description | Unit | Standard Rate | Preference |
| --- | --- | --- | --- | --- | --- |
| 0101 |  | LIVE HORSES, ASSES, MULES AND HINNIES |  |  |  |
|  | - | Horses: |  |  |  |
| 0101 21 00 | -- | Pure-bred breeding animals: | u | 30% | - |
| 0101 29 | -- | Other: |  |  |  |
| 0101 29 10 | --- | Horses for polo | u | 30% | - |
| 0101 29 90 | --- | Other | u | 30% | - |
| 0101 30 | - | Asses |  |  |  |
| 0101 30 10 | --- | Pure-bred breeding animals: | u | 30% | - |
| 0101 30 20 | --- | livestock | u | 30% | - |
| 0101 30 90 | --- | Other | u | 30% | - |
| 0101 90 | -- | Other: |  |  | - |
| 0101 90 30 | --- | Mules and hinnies as livestock | u | 30% | - |
| 0101 90 90 | --- | Other | u | 30% | - |



## 5 — Combine all chunks with rich metadata

In [10]:
def combine_text_table_chunks(text_chunks, table_chunks):
    combined_chunks = []
    
    for all_text_chunk in text_chunks:
        combined_chunks.append({
            'id'      : str(uuid.uuid4()),
            'content' : all_text_chunk['details'],
            'metadata': {
                'type'      : 'text',
                'section'   : all_text_chunk['section'],
                'chapter'   : all_text_chunk['chapter']
            }
        })

    for all_table_chunk in table_chunks:
        combined_chunks.append({
            'id'      : str(uuid.uuid4()),
            'content' : all_table_chunk['content'],
            'metadata': {
                'type'      : 'table',
                'section'   : all_table_chunk['section'],
                'chapter'   : all_table_chunk['chapter'],
                'heading'   : all_table_chunk['heading']
            }
        })

    return combined_chunks

all_chunks = combine_text_table_chunks(text_chunks, table_chunks)
print(f"Combined chunks: {len(all_chunks)}")

num_table_chunks = 0
num_text_chunks = 0
for chunk in all_chunks:
    if chunk['metadata']['type'] == 'table':
        num_table_chunks += 1
    elif chunk['metadata']['type'] == 'text':
        num_text_chunks += 1

print(f"Number of table chunks: {num_table_chunks}")
print(f"Number of text chunks: {num_text_chunks}")

Combined chunks: 1346
Number of table chunks: 1184
Number of text chunks: 162


## 6 — Embed & store in ChromaDB

We use **BGE-small-en-v1.5** (33 M params, excellent retrieval quality, fast on CPU so GPU VRAM is free for the LLM).

In [14]:
# ── load embedding model ─────────────────────────────────────────────────────
print('Loading BGE-small-en-v1.5 embedding model …')
embed_model = SentenceTransformer(
    'BAAI/bge-small-en-v1.5',
    device='cpu',          # keep GPU free for generative models
)
print('  ✅ Embedding model ready')


class BGEEmbeddingFunction(embedding_functions.EmbeddingFunction):
    """Thin wrapper so ChromaDB can call our SentenceTransformer."""
    def __call__(self, texts: List[str]) -> List[List[float]]:
        # BGE models perform better with this query prefix for retrieval
        prefixed = [f'Represent this sentence: {t}' for t in texts]
        return embed_model.encode(
            prefixed, batch_size=32, show_progress_bar=False,
            normalize_embeddings=True).tolist()


bge_ef = BGEEmbeddingFunction()

# ── create / open ChromaDB collection ────────────────────────────────────────
# Connect to ChromaDB
chroma_client = chromadb.HttpClient(host="localhost", port=8000)


# delete stale collection if re-running
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=bge_ef,
    metadata={'hnsw:space': 'cosine'},
)

# ── upsert in batches ─────────────────────────────────────────────────────────
BATCH = 64
print(f'Upserting {len(all_chunks)} chunks into ChromaDB …')
for i in range(0, len(all_chunks), BATCH):
    batch = all_chunks[i:i + BATCH]
    collection.upsert(
        ids        = [c['id'] for c in batch],
        documents  = [c['content'] for c in batch],
        metadatas  = [c['metadata'] for c in batch],
    )
    print(f'  Upserted {min(i + BATCH, len(all_chunks))} / {len(all_chunks)}')

print(f'\n✅ ChromaDB collection "{COLLECTION_NAME}" ready '
      f'({collection.count()} docs) at {PERSIST_DIR}')

Loading BGE-small-en-v1.5 embedding model …
  ✅ Embedding model ready
Upserting 1346 chunks into ChromaDB …
  Upserted 64 / 1346
  Upserted 128 / 1346
  Upserted 192 / 1346
  Upserted 256 / 1346
  Upserted 320 / 1346
  Upserted 384 / 1346
  Upserted 448 / 1346
  Upserted 512 / 1346
  Upserted 576 / 1346
  Upserted 640 / 1346
  Upserted 704 / 1346
  Upserted 768 / 1346
  Upserted 832 / 1346
  Upserted 896 / 1346
  Upserted 960 / 1346
  Upserted 1024 / 1346
  Upserted 1088 / 1346
  Upserted 1152 / 1346
  Upserted 1216 / 1346
  Upserted 1280 / 1346
  Upserted 1344 / 1346
  Upserted 1346 / 1346

✅ ChromaDB collection "hs_code_rag" ready (1346 docs) at ../content/db


## 7 — Load answer-generation LLM

In [15]:
from dotenv import load_dotenv
import os

# Load env file
load_dotenv()

api_key = os.getenv('GEMINI_API_KEY')

## 8 — Retrieval helpers

In [16]:
def retrieve(
    query: str,
    top_k: int = TOP_K,
    filter_type: str = None,   # 'text' | 'table' | None
) -> List[Dict]:
    """Retrieve top-k chunks from ChromaDB for a query."""
    where = {'type': filter_type} if filter_type else None
    results = collection.query(
        query_texts=[f'Represent this sentence: {query}'],
        n_results=top_k,
        where=where,
        include=['documents', 'metadatas', 'distances'],
    )
    chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        chunks.append({
            'content' : doc,
            'metadata': meta,
            'score'   : round(1 - dist, 4),   # cosine similarity
        })
    return chunks


def format_context(chunks: List[Dict]) -> str:
    """Format retrieved chunks as a numbered context block."""
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c['metadata']
        label = (
            f'[{i}] [{m["type"].upper()} | section {m["section"]} | chapter {m["chapter"]} | '
            f'score {c["score"]}]'
        )
        parts.append(f'{label}\n{c["content"]}')
    return '\n\n'.join(parts)


print('✅ Retrieval helpers ready')

✅ Retrieval helpers ready


## 9 — RAG answer generation

In [17]:
import requests

def generate_answer(
    query: str,
    context: str,
    # max_new_tokens: int = 512,
    # temperature: float = 0.2,
) -> str:
    """Generate an answer via chat completions."""

    system_instruction = (
        "You are a precise document assistant. "
        "Answer the user's question using ONLY the provided context. "
        "If the answer is not in the context, say: "
        "'I don't know based on the provided document.' "
        "Be concise and cite the Section and Chapter number where relevant."
    )

    user_msg = f"""Context:
    {context}

    Question: {query}"""

    payload = {
        "system_instruction": {"parts": [{"text": system_instruction}]},
        "contents": [{"parts": [{"text": user_msg}]}],
        "generationConfig": {"temperature": 0.2},
    }
    
    response = requests.post(
        "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent",
        params={"key": api_key},
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=20,
    )
    response.raise_for_status()
    payload_data = response.json()
    return payload_data["candidates"][0]["content"]["parts"][0]["text"].strip()


def ask(
    query: str,
    top_k: int = TOP_K,
    filter_type: str = None,   # 'text' | 'table' | 'image' | None
    verbose: bool = True,
) -> str:
    """End-to-end RAG: retrieve → format → generate."""
    chunks  = retrieve(query, top_k=top_k, filter_type=filter_type)
    context = format_context(chunks)
    answer  = generate_answer(query, context)

    if verbose:
        print('=' * 70)
        print(f'❓ Question: {query}')
        print('-' * 70)
        print('📚 Retrieved context:')
        for c in chunks:
            m = c['metadata']
            snippet = c['content'][:120].replace('\n', ' ')
            print(f'  [{m["type"].upper()} s.{m["section"]} c.{m["chapter"]} | score {c["score"]}] {snippet}…')
        print('-' * 70)
        print(f'💡 Answer:\n{answer}')
        print('=' * 70)

    return answer


print('✅ OpenAI RAG pipeline ready — call ask("your question")')

✅ OpenAI RAG pipeline ready — call ask("your question")


## 10 — Sample Q&A

In [18]:
# ── table-only question ───────────────────────────────────────────────────────
_ = ask('tariff percentage of horses for polo', filter_type='table')

❓ Question: tariff percentage of horses for polo
----------------------------------------------------------------------
📚 Retrieved context:
  [TABLE s.Section I c.Chapter 1 | score 0.7529] Section I - Chapter 1 - 0101  LIVE HORSES, ASSES, MULES AND HINNIES  [Table Markdown] | HS Code | Heading Indicator | De…
  [TABLE s.Section XI c.Chapter 51 | score 0.6943] Section XI - Chapter 51 - 5110  YARN OF COARSE ANIMAL HAIR OR OF HORSE HAIR (INCLUDING GIMPED HORSEHAIR YARN), WHETHER O…
  [TABLE s.Section VIII c.Chapter 41 | score 0.6814] Section VIII - Chapter 41 - 4101  RAW HIDES AND SKINS OF BOVINE (INCLUDING BUFFALO) OR EQUINE ANIMALS (FRESH OR SALTED, …
  [TABLE s.Section VIII c.Chapter 41 | score 0.6772] Section VIII - Chapter 41 - 4104  TANNED OR CRUST HIDES AND SKINS OF BOVINE (INCLUDING BUFFALO) OR EQUINE ANIMALS, 291 W…
  [TABLE s.Section II c.Chapter 5 | score 0.6748] Section II - Chapter 5 - 0507  IVORY, TORTOISE-SHELL, WHALEBONE AND WHALEBONE HAIR, HORNS, ANTLERS, HOOVES, NAILS, 

In [19]:
# ── table-only question ───────────────────────────────────────────────────────
_ = ask('hs code for internal combustion engines (ICE)', filter_type='table')

❓ Question: hs code for internal combustion engines (ICE)
----------------------------------------------------------------------
📚 Retrieved context:
  [TABLE s.Section XVI c.Chapter 84 | score 0.7779] Section XVI - Chapter 84 - 8412  OTHER ENGINES AND MOTORS  [Table Markdown] | HS Code | Heading Indicator | Description …
  [TABLE s.Section XVI c.Chapter 84 | score 0.7731] Section XVI - Chapter 84 - 8408  COMPRESSION-IGNITION INTERNAL COMBUSTION PISTON ENGINES (DIESEL OR SEMI-DIESEL ENGINES)…
  [TABLE s.Section XVI c.Chapter 84 | score 0.7546] Section XVI - Chapter 84 - 8411  TURBO-JETS, TURBO-PROPELLERS AND OTHER GAS TURBINES  [Table Markdown] | HS Code | Headi…
  [TABLE s.Section XVI c.Chapter 84 | score 0.75] Section XVI - Chapter 84 - 8481  TAPS, COCKS, VALVES AND SIMILAR APPLIANCES FOR PIPES, BOILER SHELLS, TANKS, VATS OR THE…
  [TABLE s.Section XV c.Chapter 73 | score 0.7492] Section XV - Chapter 73 - 7322  RADIATORS FOR CENTRAL HEATING, NOT ELECTRICALLY HEATED, AND PARTS THEREOF

In [20]:
# ── table-only question ───────────────────────────────────────────────────────
_ = ask('which section and chapter talks about tarrif for live animals', filter_type='text')

❓ Question: which section and chapter talks about tarrif for live animals
----------------------------------------------------------------------
📚 Retrieved context:
  [TEXT s.Section I c.Chapter 1 | score 0.738] SECTION-I CHAPTER-1 THE FIRST SCHEDULE TO THE CUSTOMS TARIFF ACT, 1975 SECTION I LIVE ANIMALS; ANIMAL PRODUCTS NOTES: 1.…
  [TEXT s.Section I c.Chapter 2 | score 0.6886] SECTION-I CHAPTER-2 CHAPTER 2 Meat and edible meat offal NOTE : This Chapter does not cover : (a) products of the kinds …
  [TEXT s.Section II c.Chapter 5 | score 0.6867] SECTION-I CHAPTER-5 CHAPTER 5 Products of animal origin, not elsewhere specified or included NOTES : 1. This Chapter doe…
  [TEXT s.Section VIII c.Chapter 41 | score 0.6803] SECTION-VIII CHAPTER-41 SECTION VIII RAW HIDES AND SKINS, LEATHER, FURSKINS AND ARTICLES THEREOF; SADDLERY AND HARNESS; …
  [TEXT s.Section XI c.Chapter 50 | score 0.6647] SECTION XI TEXTILES AND TEXTILE ARTICLES NOTES : 1. This Section does not cover : (a) animal brush m

## 10 — Inspect the ChromaDB collection

In [21]:
import pandas as pd

peek = collection.peek(limit=20)
rows = []
for doc, meta in zip(peek['documents'], peek['metadatas']):
    rows.append({
        'type'   : meta.get('type', '—'),
        'section'   : meta.get('section', '—'),
        'chapter' : meta.get('chapter', '—'),
        'snippet': doc[:100].replace('\n', ' '),
    })

df = pd.DataFrame(rows)
print(f'Collection count: {collection.count()}\n')
df

Collection count: 1346



,type,section,chapter,snippet
0,text,Section I,—,FOR OFFICIAL USE OF DGCI&S ONLY INDIAN TRADE C...
1,text,Section I,Chapter 1,SECTION-I CHAPTER-1 THE FIRST SCHEDULE TO THE ...
2,text,Section I,Chapter 2,SECTION-I CHAPTER-2 CHAPTER 2 Meat and edible ...
3,text,Section I,Chapter 3,SECTION-I CHAPTER-3 CHAPTER 3 Fish and crustac...
4,text,Section I,Chapter 4,CHAPTER 4 Dairy produce; birds’ eggs; natural ...
5,text,Section I,Chapter 4,preserved (generally SUB-HEADING NOTES : 1. Fo...
6,text,Section II,Chapter 5,SECTION-I CHAPTER-5 CHAPTER 5 Products of anim...
7,text,Section II,Chapter 6,SECTION-II CHAPTER-6 SECTION II VEGETABLE PROD...
8,text,Section II,Chapter 7,SECTION-II CHAPTER-7 CHAPTER 7 Edible vegetabl...
9,text,Section II,Chapter 8,SECTION-II CHAPTER-8 CHAPTER 8 Edible fruit an...


---
## Architecture summary

```
PDF
 ├── Text pages  ──→ Character chunking ────────────────────────────────┐
 └── Tables      ──→ Markdown ──→ Chapter's headings as summary ────────┤
                                                                        │
                                                                        ▼
                                All chunks + metadata ──────→  BGE-small-en embeddings
                                                                        │
                                                                        ▼
                                                                ChromaDB (persistent)
                                                                        │
                                Query ─────→ cosine retrieval (top-k) ──┘
                                                        │
                                                        ▼
                                                    Gemini AI ────────→ Answer
```

**VRAM budget on a T4 (15 GB):**

| Model | VRAM |
|---|---|
| BLIP captioner (fp16) | ~1.8 GB |
| BGE embedder | CPU only |
| **Total peak** | **2 GB** |

This leaves ample room on a 15 GB T4.